# Kaggle runs: Phase 0 (LLM-Check with a small model) and Phase 1 check

**How to run:** in the right-hand panel set *Accelerator = GPU T4 x2* (or T4 x1) and *Internet = On*, then click **Run All**. Nothing else to edit.

Total time: about 10 to 15 minutes. When it finishes, the results are in the *Output* panel under `repo/results/` and in `results.zip`.

In [ ]:
# Step 1: download the project code and install the few missing packages.
import os
BRANCH = "claude/project-thread-ub5z89"   # change to "main" once the PR is merged
os.chdir("/kaggle/working" if os.path.isdir("/kaggle/working") else "/content")
if not os.path.isdir("repo"):
    !git clone -q -b {BRANCH} https://github.com/wolfykulfi/LLM_Check_Hallucination_Detection.git repo
os.chdir("repo")
!git pull -q
!pip install -q "transformers>=4.45" accelerate pyyaml spacy pyarrow
!python -m spacy download en_core_web_sm -q
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Step 2 (Phase 0): score 200 RAGTruth answers with the original LLM-Check scores.
# Qwen2.5-3B needs no Hugging Face token. About 5 to 10 minutes on a T4.
CONFIG = "configs/phase0_qwen25_3b.yaml"
!python -m rag_halluc.repro.phase0_llmcheck --config {CONFIG}

In [ ]:
# Step 3: show the results table and the plot.
import glob, pandas as pd
from IPython.display import Image, display
out = sorted(glob.glob("results/phase0/*/auroc.csv"), key=os.path.getmtime)[-1]
table = pd.read_csv(out)
print("Results file:", out)
display(table[~table.score.str.contains("_L")])
display(table[table.score.str.startswith("attn_L")].sort_values("auroc_best", ascending=False).head(5))
display(Image(out.replace("auroc.csv", "attn_auroc_by_layer.png")))

In [ ]:
# Step 4 (Phase 1 check, CPU, about 3 minutes): build the sentence tables and check
# that sentence-to-token mapping works with a real model tokenizer.
!python -m rag_halluc.data.build_ragtruth --check_reader qwen2.5-1.5b

In [ ]:
# Step 5: pack all small result files into one zip to download (Output panel, results.zip).
!zip -qr ../results.zip results && ls -la ../results.zip

### Optional: Llama-3.2-3B instead of Qwen
Llama is gated. Accept the licence on https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct, create a token at https://huggingface.co/settings/tokens, add it in Kaggle under *Add-ons > Secrets* with the name `HF_TOKEN`, then run the cell below.

In [ ]:
# Optional Llama run: skipped automatically when no HF_TOKEN secret is set.
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
    have_token = True
except Exception as e:
    have_token = False
    print("No HF_TOKEN secret found, skipping the Llama run. That's fine.")
if have_token:
    !python -m rag_halluc.repro.phase0_llmcheck --config configs/phase0_llama32_3b.yaml